# 00 — Repository data inventory

A read-only stocktake for `AppleShay/thesis-best-paper-trajectories`.

**Question:** What data do we have, where are they stored, and which stage has each dataset reached?

This notebook scans local files at run time; counts are not copied from old notebook outputs. It does not call OpenAlex, scrape websites, modify source data, or build the final author-level master table. No API key is needed.

Suggested placement: `conf_paper/conf_notebooks/00_data_inventory.ipynb`. Run against your current local checkout; uncommitted local files will also be included. This is a local inventory, not a claim that every discovered file is tracked by Git.

## Interpretation rules
- The DBLP proceedings pool is **candidate control data**, not confirmed non-award papers or non-awardee authors.
- Legacy thesis files labelled `control` are a separate workflow; filenames alone do not establish non-award status.
- File existence, populated columns, and complete/valid data are different things.
- Citation totals accumulated by papers from a publication year are not citations received in that year.
- Pickle files are listed but never deserialized. Nested affiliations/authorships are flagged for inspection, not assumed complete.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_columns', 30)

ROOT_OVERRIDE = None  # e.g. r'B:\Semester 4 UU\thesis-best-paper-trajectories'
CHUNK_SIZE = 50_000
HASH_FILES = True  # detects exact duplicate files; adds one sequential read
EXPORT_REPORTS = False  # optional: writes NEW inventory reports only

def find_root():
    if ROOT_OVERRIDE:
        root = Path(ROOT_OVERRIDE).expanduser().resolve()
        if not (root / 'data').is_dir() or not (root / 'conf_paper').is_dir():
            raise ValueError('ROOT_OVERRIDE must point to the repository root.')
        return root
    cwd = Path.cwd().resolve()
    for p in (cwd, *cwd.parents):
        if (p / 'data').is_dir() and (p / 'conf_paper').is_dir():
            return p
    raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')

ROOT = find_root()
SCAN_ROOTS = [ROOT / 'data', ROOT / 'conf_paper' / 'conf_data',
              ROOT / 'conf_paper' / 'conf_notebooks' / 'output']
REPORT_DIR = ROOT / 'conf_paper' / 'conf_data' / 'data_inventory_reports'
RUN_AT = datetime.now(timezone.utc).isoformat()

def git_value(args):
    try:
        result = subprocess.run(['git', '-C', str(ROOT), *args], capture_output=True,
                                text=True, timeout=10)
        return result.stdout.strip() if result.returncode == 0 else None
    except (OSError, subprocess.TimeoutExpired):
        return None

display(pd.DataFrame([{'repository_root': str(ROOT), 'run_at_utc': RUN_AT,
                       'git_commit': git_value(['rev-parse', 'HEAD']),
                       'git_branch': git_value(['branch', '--show-current'])}]))

,repository_root,run_at_utc,git_commit,git_branch
0,B:\Semester 4 UU\thesis-best-paper-trajectories,2026-10-29T10:47:20.151818+00:00,None,None


## 1. Key data locations and intended roles

These descriptions come from the repository workflow reviewed when this notebook was prepared. The scan below checks what actually exists now. “Expected role” is not certification of data quality or cohort membership.

In [2]:
KEY_FILES = {
    'conf_paper/conf_data/huang_matched_openalex.csv': ('award', 'Award corpus copied into the paper workflow; compare with thesis copy.'),
    'data/matched/huang_matched_openalex.csv': ('award', 'Original enriched award corpus; paper IDs, conference/year and nested authorships.'),
    'conf_paper/conf_data/01_unmatched_papers.csv': ('award', 'Award records without accepted OpenAlex IDs.'),
    'conf_paper/conf_data/02_award_authors_long.csv': ('award', 'Flattened award author-paper records; not unique authors.'),
    'conf_paper/conf_data/02_award_authors_with_career_stage.csv': ('award', 'Author-paper records with career age and junior/senior labels; requires validation.'),
    'conf_paper/conf_data/02_author_profiles.csv': ('award', 'Author profile snapshots and estimated first-publication years.'),
    'conf_paper/conf_data/04_all_works_cache.pkl': ('award', 'Cached author works. Listed only; never unpickled here.'),
    'conf_paper/conf_data/04_author_before_after_lift.csv': ('award', 'Before/after summary: reviewed notebook uses ±3 years and lifetime citations to publication cohorts.'),
    'conf_paper/conf_data/04_trajectory_long.csv': ('award', 'Relative-year trajectories; trace citation construction before treating as annual citation flow.'),
    'conf_paper/conf_notebooks/output/dblp_papers_raw.csv': ('DBLP candidates', 'Proceedings pool; award winners are not automatically excluded.'),
    'conf_paper/conf_notebooks/output/dblp_main_volumes_log.csv': ('DBLP candidates', 'Proceedings-volume collection log; inspect missing venues/years.'),
    'conf_paper/conf_notebooks/output/dblp_openalex_checkpoint.csv': ('DBLP matching', 'Resumable paper matching state; distinguish accepted, ambiguous, unmatched and API errors.'),
    'conf_paper/conf_notebooks/output/dblp_openalex_matched.csv': ('DBLP matching', 'Exported accepted matches; may lag behind checkpoint.'),
    'conf_paper/conf_notebooks/output/dblp_openalex_ambiguous.csv': ('DBLP matching', 'Exported candidates requiring manual review.'),
    'conf_paper/conf_notebooks/output/dblp_openalex_unmatched.csv': ('DBLP matching', 'Exported unresolved/error records; not proof papers are absent from OpenAlex.'),
    'conf_paper/conf_notebooks/output/dblp_openalex_match_report.csv': ('DBLP matching', 'Exported coverage report; may be stale after an interrupted run.'),
    'data/matched/matched_pairs_clean.csv': ('legacy comparisons', 'Thesis comparison pairs; audit provenance before calling these non-awardees.'),
    'data/matched/nonaward_papers_raw.csv': ('legacy comparisons', 'Separate legacy paper comparison pool; not the large DBLP collection.'),
    'data/matched/all_authors_within_paper.csv': ('legacy award authors', 'Within-paper author records; numeric authorship positions and yearly fields.'),
    'data/matched/junior_authors_all_conferences.csv': ('legacy award authors', 'Junior author-paper mapping, including institutions where captured.'),
    'data/matched/award_to_citing_edges.csv': ('citation network', 'Forward edges from award papers to citing papers.'),
    'data/matched/citing_papers.csv': ('citation network', 'Forward-citing paper metadata, years and author IDs; may exist only locally.'),
    'data/matched/cited_papers.csv': ('citation network', 'Backward references: NOT interchangeable with citing_papers.csv.'),
    'data/derived/junior_awardpaper_self_cites_summary.csv': ('self-citations', 'Legacy junior-author self-citation summary; not necessarily restricted to +5 years.'),
    'data/cd_trajectory/author_papers_panel.csv': ('legacy panel', 'Author-paper panel for CD analyses; group definitions follow upstream matching.')
}
ledger = pd.DataFrame([{'path': p, 'group': g, 'expected_role': r,
                       'exists_locally': (ROOT / p).is_file()}
                      for p, (g, r) in KEY_FILES.items()])
display(ledger)

,path,group,expected_role,exists_locally
0,conf_paper/conf_data/huang_matched_openalex.csv,award,Award corpus copied into the paper workflow; compare with thesis copy.,True
1,data/matched/huang_matched_openalex.csv,award,"Original enriched award corpus; paper IDs, conference/year and nested authorships.",True
2,conf_paper/conf_data/01_unmatched_papers.csv,award,Award records without accepted OpenAlex IDs.,True
3,conf_paper/conf_data/02_award_authors_long.csv,award,Flattened award author-paper records; not unique authors.,True
4,conf_paper/conf_data/02_award_authors_with_career_stage.csv,award,Author-paper records with career age and junior/senior labels; requires validation.,True
5,conf_paper/conf_data/02_author_profiles.csv,award,Author profile snapshots and estimated first-publication years.,True
6,conf_paper/conf_data/04_all_works_cache.pkl,award,Cached author works. Listed only; never unpickled here.,True
7,conf_paper/conf_data/04_author_before_after_lift.csv,award,Before/after summary: reviewed notebook uses ±3 years and lifetime citations to publication coho...,True
8,conf_paper/conf_data/04_trajectory_long.csv,award,Relative-year trajectories; trace citation construction before treating as annual citation flow.,True
9,conf_paper/conf_notebooks/output/dblp_papers_raw.csv,DBLP candidates,Proceedings pool; award winners are not automatically excluded.,True


## 2. Discover files and profile every CSV

CSV profiling reads all rows in chunks, using strings to avoid losing identifier formatting. The inventory includes row counts, columns, missing/blank values, file sizes and UTC modification times. Modification time is a filesystem timestamp, not the collection date.

JSON and binary caches are listed without displaying their contents. This notebook deliberately does not print keys, credentials, or raw cache contents. CSV previews later are restricted to selected ordinary metadata fields.

In [3]:
DATA_SUFFIXES = {'.csv', '.tsv', '.json', '.jsonl', '.pkl', '.pickle', '.parquet',
                 '.feather', '.xlsx', '.xls', '.sqlite', '.db', '.gz', '.zip'}

def group_for(rel):
    if rel in KEY_FILES:
        return KEY_FILES[rel][0]
    if rel.startswith('conf_paper/conf_notebooks/output/dblp_'):
        return 'DBLP matching / candidates'
    if rel.startswith('conf_paper/conf_data/'):
        return 'paper workflow: other'
    if rel.startswith('conf_paper/conf_notebooks/output/'):
        return 'source resolution / other'
    return 'legacy thesis: ' + Path(rel).parts[1]

def file_digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def profile_csv(path, root):
    rel = path.relative_to(root).as_posix()
    rows = 0
    populated = Counter()
    cols = []
    for chunk in pd.read_csv(path, dtype=str, chunksize=CHUNK_SIZE,
                             encoding='utf-8-sig', keep_default_na=False):
        cols = list(chunk.columns)
        rows += len(chunk)
        for col in cols:
            populated[col] += int(chunk[col].fillna('').str.strip().ne('').sum())
    if not cols:
        cols = list(pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
    fields = [{'path': rel, 'column': c, 'rows': rows,
               'nonempty_rows': populated[c], 'missing_or_blank_rows': rows - populated[c],
               'nonempty_pct': round(100 * populated[c] / rows, 2) if rows else None}
              for c in cols]
    return rows, cols, fields

found = set()
for base in SCAN_ROOTS:
    if base.exists():
        for p in base.rglob('*'):
            if p.is_file() and p.suffix.lower() in DATA_SUFFIXES:
                if REPORT_DIR not in p.parents and not p.is_symlink():
                    found.add(p)

file_rows, column_rows = [], []
for p in sorted(found):
    rel = p.relative_to(ROOT).as_posix()
    stat = p.stat()
    row = {'path': rel, 'group': group_for(rel), 'format': p.suffix.lower(),
           'size_mb': round(stat.st_size / 1024**2, 3),
           'modified_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(),
           'rows': None, 'n_columns': None, 'columns': None,
           'sha256': None, 'profile_status': 'listed only; not deserialized', 'error': None}
    try:
        if p.suffix.lower() == '.csv':
            n, cols, fields = profile_csv(p, ROOT)
            row.update(rows=n, n_columns=len(cols), columns=' | '.join(cols), profile_status='CSV fully scanned')
            column_rows.extend(fields)
        if HASH_FILES:
            row['sha256'] = file_digest(p)
    except Exception as exc:
        row.update(profile_status='ERROR: inspect file', error=type(exc).__name__)
    file_rows.append(row)

inventory = pd.DataFrame(file_rows, columns=['path', 'group', 'format', 'size_mb',
    'modified_utc', 'rows', 'n_columns', 'columns', 'sha256', 'profile_status', 'error'])
columns_df = pd.DataFrame(column_rows, columns=['path', 'column', 'rows',
    'nonempty_rows', 'missing_or_blank_rows', 'nonempty_pct'])
for c in ['rows', 'n_columns']:
    inventory[c] = inventory[c].astype('Int64')
print(f'Found {len(inventory):,} data files; scanned {inventory.rows.notna().sum():,} CSVs.')
display(inventory[['path', 'group', 'size_mb', 'rows', 'n_columns', 'profile_status']])

Found 103 data files; scanned 89 CSVs.


,path,group,size_mb,rows,n_columns,profile_status
0,conf_paper/conf_data/01_exploration_summary.csv,paper workflow: other,0.000,8,2,CSV fully scanned
1,conf_paper/conf_data/01_unmatched_papers.csv,award,0.009,22,10,CSV fully scanned
2,conf_paper/conf_data/02_author_profiles.csv,award,0.188,2701,8,CSV fully scanned
3,conf_paper/conf_data/02_author_profiles_raw.csv,paper workflow: other,0.175,2701,7,CSV fully scanned
4,conf_paper/conf_data/02_award_authors_long.csv,award,0.490,3150,7,CSV fully scanned
...,...,...,...,...,...,...
98,data/raw/icwsm_awards_raw.csv,legacy thesis: raw,0.006,37,7,CSV fully scanned
99,data/raw/icwsm_jcdl_author_profiles.csv,legacy thesis: raw,0.085,398,14,CSV fully scanned
100,data/raw/icwsm_jcdl_awards_combined.csv,legacy thesis: raw,0.008,51,7,CSV fully scanned
101,data/raw/icwsm_jcdl_awards_raw.csv,legacy thesis: raw,0.034,112,11,CSV fully scanned


### Location summary, missing expected artifacts, and identical copies

An identical hash means byte-for-byte identical files. Different hashes do not necessarily mean different records: ordering or formatting can also differ. Group row totals are deliberately not summed because files can overlap or be intermediate copies.

In [4]:
if not inventory.empty:
    display(inventory.groupby('group', dropna=False).agg(files=('path', 'size'),
            total_size_mb=('size_mb', 'sum')).reset_index())
display(ledger.loc[~ledger.exists_locally, ['path', 'expected_role']])
if HASH_FILES:
    hashes = inventory.dropna(subset=['sha256'])
    duplicates = hashes.loc[hashes.duplicated('sha256', keep=False)]
    display(duplicates[['sha256', 'path', 'size_mb']].sort_values(['sha256', 'path']))
display(inventory.loc[inventory.error.notna(), ['path', 'error']])

,group,files,total_size_mb
0,DBLP candidates,2,11.013
1,DBLP matching,5,1.143
2,award,9,32.728
3,citation network,3,383.467
4,legacy award authors,2,3.272
5,legacy comparisons,2,0.215
6,legacy panel,1,2.459
7,legacy thesis: cd_trajectory,4,2.501
8,legacy thesis: cleaned,2,0.794
9,legacy thesis: figures,10,0.015


,path,expected_role


,sha256,path,size_mb
57,4e01b86352628836facc6db163ab98f63c22ae3cd9329871b9a5ee6277db530b,data/matched/award_to_cited_edges.csv,2.522
63,4e01b86352628836facc6db163ab98f63c22ae3cd9329871b9a5ee6277db530b,data/matched/cited_refs_checkpoint.csv,2.522
13,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421d16b75b389f5be4bdd,conf_paper/conf_data/huang_matched_openalex.csv,2.754
71,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421d16b75b389f5be4bdd,data/matched/huang_matched_openalex.csv,2.754
18,9024ab566d7f6f018310e42e924250717edd65d2e2683c95a5c089c7b47bc81a,conf_paper/conf_data/nb07a_venue_year_source_lookup.csv,0.033
21,9024ab566d7f6f018310e42e924250717edd65d2e2683c95a5c089c7b47bc81a,conf_paper/conf_data/nb07b_fix_venue_year_source_lookup.csv,0.033
58,d073ceab419a54f446f87b114858f13ce738f50008e91b00e42f6e6a7bcd30d2,data/matched/award_to_citing_edges.csv,37.285
64,d073ceab419a54f446f87b114858f13ce738f50008e91b00e42f6e6a7bcd30d2,data/matched/citing_edges_checkpoint.csv,37.285


,path,error
69,data/matched/control_authors.csv,EmptyDataError
70,data/matched/control_progress.csv,EmptyDataError


## 3. Award data: available fields and coverage

The tables below inspect actual schema and populated values; they do not infer that every author is correctly disambiguated or that every institution is historical. A nonempty `authorships` string may still contain missing institutions, and lifetime totals are not five-year totals.

In [5]:
def load_csv(rel):
    p = ROOT / rel
    if not p.exists():
        return None
    try:
        return pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    except Exception as exc:
        print(f'Cannot read {rel}: {type(exc).__name__}')
        return None

def present(series):
    return series.fillna('').astype(str).str.strip().ne('')

def short_id(series):
    return series.fillna('').astype(str).str.strip().str.extract(r'([AW]\d+)$', expand=False)

AWARD_FILES = [p for p, (g, _) in KEY_FILES.items()
               if g in {'award', 'legacy award authors'} and p.endswith('.csv')]
display(inventory.loc[inventory.path.isin(AWARD_FILES), ['path', 'rows', 'n_columns', 'columns']])
important = ['author_id', 'author_name', 'paper_openalex_id', 'openalex_id', 'year',
             'award_year', 'conference', 'first_pub_year', 'career_age_at_award',
             'career_stage', 'seniority', 'author_position', 'position', 'institutions',
             'works_count', 'cited_by_count', 'authorships', 'counts_by_year']
display(columns_df.loc[columns_df.path.isin(AWARD_FILES) & columns_df.column.isin(important)])

award_summary = []
for rel in ['conf_paper/conf_data/02_award_authors_with_career_stage.csv',
            'data/matched/huang_matched_openalex.csv',
            'conf_paper/conf_data/huang_matched_openalex.csv']:
    df = load_csv(rel)
    if df is None:
        continue
    s = {'path': rel, 'rows': len(df)}
    if 'author_id' in df:
        s['unique_author_ids'] = short_id(df.author_id).nunique()
    for pc in ['paper_openalex_id', 'openalex_id']:
        if pc in df:
            s['unique_paper_ids'] = short_id(df[pc]).nunique()
            break
    yc = next((c for c in ['award_year', 'year'] if c in df), None)
    if yc:
        years = pd.to_numeric(df[yc], errors='coerce')
        s.update(year_min=years.min(), year_max=years.max(),
                 rows_outside_2000_2018=int((years.notna() & ~years.between(2000, 2018)).sum()),
                 missing_year=int(years.isna().sum()))
    award_summary.append(s)
    if 'career_stage' in df:
        display(df.career_stage.replace('', '(missing)').value_counts(dropna=False).rename_axis('career_stage').to_frame('author_paper_rows'))
    if 'career_age_at_award' in df:
        age = pd.to_numeric(df.career_age_at_award, errors='coerce')
        print(rel, {'missing_career_age': int(age.isna().sum()),
                    'negative_career_age': int(age.lt(0).sum()),
                    'career_age_over_60_review': int(age.gt(60).sum())})
display(pd.DataFrame(award_summary))

,path,rows,n_columns,columns
1,conf_paper/conf_data/01_unmatched_papers.csv,22,10,year | conference | paper_title | paper_url | authors | openalex_id | doi | oa_title | authorshi...
2,conf_paper/conf_data/02_author_profiles.csv,2701,8,author_id | display_name | works_count | cited_by_count | h_index | i10_index | first_pub_year_c...
4,conf_paper/conf_data/02_award_authors_long.csv,3150,7,paper_openalex_id | paper_title | conference | award_year | author_id | author_name | author_pos...
5,conf_paper/conf_data/02_award_authors_with_career_stage.csv,3150,14,paper_openalex_id | paper_title | conference | award_year | author_id | author_name | author_pos...
9,conf_paper/conf_data/04_author_before_after_lift.csv,2700,15,author_id | author_name | career_stage | award_year | first_pub_year | pubs_before | pubs_after ...
12,conf_paper/conf_data/04_trajectory_long.csv,23721,5,author_id | treatment | rel_year | works | citations
13,conf_paper/conf_data/huang_matched_openalex.csv,925,10,year | conference | paper_title | paper_url | authors | openalex_id | doi | oa_title | authorshi...
52,data/matched/all_authors_within_paper.csv,2291,10,award_id | conference | award_year | author_id | author_name | position | career_age | total_pub...
71,data/matched/huang_matched_openalex.csv,925,10,year | conference | paper_title | paper_url | authors | openalex_id | doi | oa_title | authorshi...
72,data/matched/junior_authors_all_conferences.csv,603,14,award_id | conference | award_year | work_id | award_title | author_id | author_name | author_po...


,path,column,rows,nonempty_rows,missing_or_blank_rows,nonempty_pct
2,conf_paper/conf_data/01_unmatched_papers.csv,year,22,22,0,100.00
3,conf_paper/conf_data/01_unmatched_papers.csv,conference,22,22,0,100.00
7,conf_paper/conf_data/01_unmatched_papers.csv,openalex_id,22,0,22,0.00
10,conf_paper/conf_data/01_unmatched_papers.csv,authorships,22,0,22,0.00
12,conf_paper/conf_data/02_author_profiles.csv,author_id,2701,2701,0,100.00
14,conf_paper/conf_data/02_author_profiles.csv,works_count,2701,2701,0,100.00
15,conf_paper/conf_data/02_author_profiles.csv,cited_by_count,2701,2701,0,100.00
19,conf_paper/conf_data/02_author_profiles.csv,first_pub_year,2701,2701,0,100.00
27,conf_paper/conf_data/02_award_authors_long.csv,paper_openalex_id,3150,3150,0,100.00
29,conf_paper/conf_data/02_award_authors_long.csv,conference,3150,3150,0,100.00


,author_paper_rows
career_stage,
Senior,2313
Junior,828
(missing),9


conf_paper/conf_data/02_award_authors_with_career_stage.csv {'missing_career_age': 9, 'negative_career_age': 4, 'career_age_over_60_review': 117}


,path,rows,unique_author_ids,unique_paper_ids,year_min,year_max,rows_outside_2000_2018,missing_year
0,conf_paper/conf_data/02_award_authors_with_career_stage.csv,3150,2710.0,857,2000,2018,0,0
1,data/matched/huang_matched_openalex.csv,925,NaN,903,2000,2018,0,0
2,conf_paper/conf_data/huang_matched_openalex.csv,925,NaN,903,2000,2018,0,0


## 4. DBLP candidate papers: size, schema and venue/year coverage

This section does not label the proceedings pool as non-winning. It checks its actual rows and duplicate metadata keys. A populated author-name string is not a resolved OpenAlex author identity.

In [6]:
DBLP_RAW = 'conf_paper/conf_notebooks/output/dblp_papers_raw.csv'
raw = load_csv(DBLP_RAW)
if raw is None:
    print('DBLP source file not found locally.')
else:
    display(pd.DataFrame([{'path': DBLP_RAW, 'rows': len(raw), 'columns': ' | '.join(raw.columns)}]))
    preview_cols = [c for c in ['conference', 'year', 'title', 'authors', 'pages'] if c in raw]
    display(raw[preview_cols].head(8))
    if {'conference', 'year'}.issubset(raw.columns):
        raw['conference_normalized'] = raw.conference.str.strip().str.upper()
        raw['year_numeric'] = pd.to_numeric(raw.year, errors='coerce')
        display(raw.groupby('conference_normalized').agg(rows=('year', 'size'),
                    first_year=('year_numeric', 'min'), last_year=('year_numeric', 'max')).reset_index())
        venue_year = raw.groupby(['conference_normalized', 'year_numeric'], dropna=False).size().rename('paper_rows').reset_index()
        display(venue_year.pivot_table(index='conference_normalized', columns='year_numeric',
                    values='paper_rows', aggfunc='sum'))
    keys = [c for c in ['conference', 'year', 'title'] if c in raw]
    if len(keys) == 3:
        print('Duplicate conference/year/title rows beyond first occurrence:', int(raw.duplicated(keys).sum()))
    print('Note: proceedings coverage is not guaranteed complete; zero/missing cells need collection-log review.')

,path,rows,columns
0,conf_paper/conf_notebooks/output/dblp_papers_raw.csv,83319,conference | year | title | authors | pages


,conference,year,title,authors,pages
0,aaai,2000,Inter-Layer Learning Towards Emergent Cooperative Behavior.,Shawn Arseneau; Wei Sun; Changpeng Zhao; Jeremy R. Cooperstock,3-8
1,aaai,2000,Coordination Failure and Congestion in Information Networks.,Ann M. Bell; William A. Sethares; James A. Bucklew,9-14
2,aaai,2000,Non-Deterministic Social Laws.,Michael H. Coen,15-21
3,aaai,2000,Solving Combinatorial Auctions Using Stochastic Local Search.,Holger H. Hoos; Craig Boutilier,22-29
4,aaai,2000,A Mechanism for Group Decision Making in Collaborative Activity.,Luke Hunsberger; Massimo Zancanaro,30-35
5,aaai,2000,Cobot in LambdaMOO: A Social Statistics Agent.,Charles Lee Isbell Jr.; Michael J. Kearns; David P. Kormann; Satinder Singh; Peter Stone,36-41
6,aaai,2000,Semantics of Agent Communication Languages for Group Interaction.,Sanjeev Kumar; Marcus J. Huber; David McGee; Philip R. Cohen; Hector J. Levesque,42-47
7,aaai,2000,Deliberation in Equilibrium: Bargaining in Computationally Complex Problems.,Kate Larson; Tuomas Sandholm,48-55


,conference_normalized,rows,first_year,last_year
0,AAAI,7004,2000,2018
1,ACL,2870,2000,2018
2,CHI,5522,2000,2018
3,CIKM,4477,2001,2018
4,CVPR,8458,2000,2018
5,FOCS,1449,2000,2018
6,FSE,58,2016,2017
7,ICCV,2741,2001,2017
8,ICML,4106,2000,2018
9,ICSE,2518,2000,2018


year_numeric,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
conference_normalized,,,,,,,,,,,,,,,,,,,
AAAI,233.0,NaN,180.0,NaN,194.0,325.0,385.0,368.0,356.0,NaN,313.0,319.0,353.0,251.0,474.0,674.0,691.0,786.0,1102.0
ACL,79.0,70.0,65.0,71.0,88.0,134.0,307.0,204.0,119.0,121.0,160.0,164.0,111.0,174.0,147.0,174.0,231.0,195.0,256.0
CHI,72.0,69.0,61.0,75.0,93.0,93.0,151.0,182.0,218.0,277.0,302.0,409.0,369.0,392.0,465.0,484.0,545.0,600.0,665.0
CIKM,NaN,92.0,101.0,88.0,94.0,163.0,140.0,134.0,240.0,337.0,323.0,428.0,451.0,382.0,270.0,244.0,327.0,350.0,313.0
CVPR,229.0,273.0,NaN,207.0,258.0,359.0,323.0,538.0,506.0,382.0,461.0,439.0,466.0,471.0,540.0,601.0,643.0,783.0,979.0
FOCS,66.0,65.0,80.0,64.0,64.0,73.0,71.0,66.0,82.0,73.0,82.0,88.0,80.0,78.0,68.0,86.0,86.0,90.0,87.0
FSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0,29.0,NaN
ICCV,NaN,104.0,NaN,NaN,NaN,NaN,NaN,389.0,NaN,308.0,NaN,339.0,NaN,454.0,NaN,526.0,NaN,621.0,NaN
ICML,151.0,80.0,87.0,117.0,117.0,134.0,140.0,150.0,157.0,179.0,159.0,152.0,243.0,283.0,310.0,270.0,322.0,434.0,621.0


Duplicate conference/year/title rows beyond first occurrence: 36
Note: proceedings coverage is not guaranteed complete; zero/missing cells need collection-log review.


## 5. OpenAlex matching progress — checkpoint versus exports

Count unique source-row IDs, not API calls. The checkpoint may be newer than the exported files. “Not in checkpoint” means not recorded as attempted, not proven absent from OpenAlex. A source-row ID is position-based in the reviewed matching notebooks, so it must also agree with conference/year/title before being trusted.

In [7]:
PREFIX = 'conf_paper/conf_notebooks/output/'
checkpoint = load_csv(PREFIX + 'dblp_openalex_checkpoint.csv')
matching_tables = []
for name in ['dblp_openalex_checkpoint.csv', 'dblp_openalex_matched.csv',
             'dblp_openalex_ambiguous.csv', 'dblp_openalex_unmatched.csv']:
    frame = checkpoint if name.endswith('checkpoint.csv') else load_csv(PREFIX + name)
    if frame is None:
        continue
    s = {'file': name, 'rows': len(frame)}
    if 'dblp_row_id' in frame:
        s['unique_source_row_ids'] = frame.loc[present(frame.dblp_row_id), 'dblp_row_id'].nunique()
        s['duplicate_source_row_ids'] = int(frame.loc[present(frame.dblp_row_id)].duplicated('dblp_row_id').sum())
    if 'match_status' in frame:
        s.update(frame.match_status.replace('', '(blank)').value_counts().to_dict())
    matching_tables.append(s)
display(pd.DataFrame(matching_tables))

matching_progress = pd.DataFrame()
if checkpoint is not None and 'dblp_row_id' in checkpoint:
    has_dupes = checkpoint.loc[present(checkpoint.dblp_row_id)].duplicated('dblp_row_id').any()
    if has_dupes:
        print('WARNING: duplicate checkpoint row IDs; conflicts must be resolved before unique-progress counts.')
    elif raw is not None and {'conference', 'year', 'title'}.issubset(raw.columns) and 'match_status' in checkpoint:
        source = raw.loc[present(raw.title) & pd.to_numeric(raw.year, errors='coerce').notna()].copy().reset_index(drop=True)
        source['dblp_row_id'] = [f'DBLP_{i:07d}' for i in range(1, len(source) + 1)]
        cp = checkpoint.loc[present(checkpoint.dblp_row_id)].copy()
        merged = source[['dblp_row_id', 'conference', 'year', 'title']].merge(
            cp, on='dblp_row_id', how='left', suffixes=('_source', '_checkpoint'), validate='one_to_one', indicator=True)
        seen = merged['_merge'].eq('both')
        agreement = pd.Series(True, index=merged.index)
        for c in ['conference', 'year', 'title']:
            a, b = c + '_source', c + '_checkpoint'
            if b not in merged:
                agreement &= False
            else:
                agreement &= merged[a].fillna('').str.strip().str.upper().eq(merged[b].fillna('').str.strip().str.upper())
        comparable = seen & agreement
        matched = comparable & merged.match_status.eq('matched')
        if 'openalex_id' in merged:
            matched &= short_id(merged.openalex_id).notna()
        else:
            matched &= False
        matching_progress = pd.DataFrame([
            {'measure': 'Raw pool after matcher input filtering', 'count': len(source)},
            {'measure': 'Recorded in checkpoint (row ID)', 'count': int(seen.sum())},
            {'measure': 'Checkpoint source-metadata disagreements', 'count': int((seen & ~agreement).sum())},
            {'measure': 'Accepted matches with ID and agreeing source metadata', 'count': int(matched.sum())},
            {'measure': 'Not recorded in checkpoint', 'count': int((~seen).sum())},
            {'measure': 'Checkpoint IDs absent from current source', 'count': int((~cp.dblp_row_id.isin(source.dblp_row_id)).sum())}
        ])
        display(matching_progress)
        print('Accepted coverage of current input: {:.2%}'.format(matched.sum() / len(source)) if len(source) else 'Empty input')
        display(merged.loc[comparable, 'match_status'].value_counts(dropna=False).rename_axis('status').to_frame('source_rows'))
        award_ids = set()
        for rel in ['data/matched/huang_matched_openalex.csv', 'conf_paper/conf_data/huang_matched_openalex.csv']:
            aw = load_csv(rel)
            if aw is not None and 'openalex_id' in aw:
                award_ids.update(short_id(aw.openalex_id).dropna())
        if award_ids and 'openalex_id' in merged:
            overlap = matched & short_id(merged.openalex_id).isin(award_ids)
            print('Accepted DBLP rows overlapping observed award work IDs:', int(overlap.sum()))
            print('This is a paper-ID overlap check only; zero does not certify non-awardee authors or complete award exclusion.')
    exported = load_csv(PREFIX + 'dblp_openalex_matched.csv')
    if exported is not None and {'dblp_row_id', 'match_status'}.issubset(checkpoint) and 'dblp_row_id' in exported:
        cp_ids = set(checkpoint.loc[checkpoint.match_status.eq('matched'), 'dblp_row_id'])
        ex_ids = set(exported.dblp_row_id)
        print({'accepted_checkpoint_ids_not_exported': len(cp_ids - ex_ids),
               'exported_ids_not_currently_accepted_in_checkpoint': len(ex_ids - cp_ids)})
report = load_csv(PREFIX + 'dblp_openalex_match_report.csv')
if report is not None:
    print('Saved coverage report (may lag behind checkpoint):')
    display(report)

,file,rows,unique_source_row_ids,duplicate_source_row_ids,matched,unmatched,ambiguous
0,dblp_openalex_checkpoint.csv,317,317,0,278.0,26.0,13.0
1,dblp_openalex_matched.csv,244,244,0,244.0,NaN,NaN
2,dblp_openalex_ambiguous.csv,6,6,0,NaN,NaN,6.0
3,dblp_openalex_unmatched.csv,25,25,0,NaN,25.0,NaN


,measure,count
0,Raw pool after matcher input filtering,83319
1,Recorded in checkpoint (row ID),317
2,Checkpoint source-metadata disagreements,0
3,Accepted matches with ID and agreeing source metadata,278
4,Not recorded in checkpoint,83002
5,Checkpoint IDs absent from current source,0


Accepted coverage of current input: 0.33%


,source_rows
status,
matched,278
unmatched,26
ambiguous,13


Accepted DBLP rows overlapping observed award work IDs: 5
This is a paper-ID overlap check only; zero does not certify non-awardee authors or complete award exclusion.
{'accepted_checkpoint_ids_not_exported': 34, 'exported_ids_not_currently_accepted_in_checkpoint': 0}
Saved coverage report (may lag behind checkpoint):


,conference,year,papers,matched,ambiguous,unmatched,mean_match_score,match_rate
0,AAAI,2000,233,205,4,24,0.9381648068669528,0.8798283261802575
1,AAAI,2002,42,39,2,1,0.9854047619047619,0.9285714285714286


## 6. Legacy comparisons, citation-network files and self-citation inputs

These files must not be silently combined with the DBLP pool. In the reviewed thesis pipeline, the `matched_pairs_clean.csv` comparison pool was drawn from award-paper authors with a seniority filter. Separate legacy non-award paper files also exist.

Self-citation work requires the **forward-citing** metadata (`citing_papers.csv`) and citation edges. Backward references (`cited_papers.csv`) are a different dataset. A stored summary is not a validated all-author five-year rate.

In [8]:
legacy = inventory.path.str.startswith('data/')
display(inventory.loc[legacy, ['path', 'group', 'rows', 'n_columns', 'columns']])

WATCH_FIELDS = {'author_id', 'treated_id', 'control_id', 'work_id', 'award_paper_id',
                'citing_paper_id', 'author_ids', 'publication_year', 'relative_year',
                'counts_by_year', 'share_self_citing', 'seniority', 'institutions'}
display(columns_df.loc[columns_df.path.str.startswith('data/') & columns_df.column.isin(WATCH_FIELDS)])

,path,group,rows,n_columns,columns
34,data/cd_trajectory/author_papers_panel.csv,legacy panel,39164,6,author_id | group | award_year | paper_id | paper_year | relative_year
35,data/cd_trajectory/author_year_cd5.csv,legacy thesis: cd_trajectory,5468,5,author_id | group | award_year | relative_year | cd5
36,data/cd_trajectory/cd5_did_summary.csv,legacy thesis: cd_trajectory,4,6,is_awardee | period | mean | median | std | count
37,data/cd_trajectory/paper_cd5_scores.csv,legacy thesis: cd_trajectory,44501,5,paper_id | paper_year | cd5 | ni | nj
38,data/cd_trajectory/papers_citation_counts.csv,legacy thesis: cd_trajectory,54072,2,paper_id | cited_by_count
...,...,...,...,...,...
98,data/raw/icwsm_awards_raw.csv,legacy thesis: raw,37,7,year | conference | paper_title | paper_url | authors | award_type | award_year
99,data/raw/icwsm_jcdl_author_profiles.csv,legacy thesis: raw,398,14,openalex_paper_id | paper_title | conference | award_type | award_year | author_id | author_name...
100,data/raw/icwsm_jcdl_awards_combined.csv,legacy thesis: raw,51,7,year | conference | paper_title | paper_url | authors | award_type | award_year
101,data/raw/icwsm_jcdl_awards_raw.csv,legacy thesis: raw,112,11,year | conference | paper_title | paper_url | authors | award_type | award_year | openalex_id | ...


,path,column,rows,nonempty_rows,missing_or_blank_rows,nonempty_pct
245,data/cd_trajectory/author_papers_panel.csv,author_id,39164,39164,0,100.00
250,data/cd_trajectory/author_papers_panel.csv,relative_year,39164,39164,0,100.00
251,data/cd_trajectory/author_year_cd5.csv,author_id,5468,5468,0,100.00
254,data/cd_trajectory/author_year_cd5.csv,relative_year,5468,5468,0,100.00
279,data/derived/junior_awardpaper_self_cites_summary.csv,award_paper_id,596,596,0,100.00
...,...,...,...,...,...,...
700,data/profiles/junior_profiles_all.csv,author_id,602,602,0,100.00
713,data/profiles/junior_profiles_all.csv,counts_by_year,602,602,0,100.00
714,data/profiles/junior_profiles_progress.csv,author_id,600,600,0,100.00
727,data/profiles/junior_profiles_progress.csv,counts_by_year,600,600,0,100.00


## 7. Supervisor requirements: where the ingredients are, and what is not established

This is a workflow checklist, not an automatically certified readiness score. Exact coverage is in the schema/missingness tables above. Do not substitute field-name matches for validation of the metric.

In [9]:
requirements = pd.DataFrame([
    ('Conference/year/paper', '02_award_authors_long.csv; huang_matched_openalex.csv', 'Join keys exist in reviewed workflow; verify cohort and unresolved paper IDs.'),
    ('Authors', '02_award_authors_long.csv; 02_author_profiles.csv', 'Author-paper rows are not unique authors. Resolve missing/deprecated IDs.'),
    ('Seniority at award', '02_award_authors_with_career_stage.csv', 'Labels exist; audit suspicious first-publication years and negative career ages.'),
    ('Institution at award', 'Nested award-paper authorships; legacy junior institutions field', 'Not retained in journal flattened table. Paper affiliations are a proxy, not current author affiliations.'),
    ('Authorship position', '02_award_authors_long.csv; legacy position fields', 'First/middle/last exists; exact order needs extraction from the paper authorship list.'),
    ('Publication totals by year, ±5', '04_all_works_cache.pkl; 04_trajectory_long.csv', 'Recompute per reference event. Before/after summary notebook uses ±3, not ±5.'),
    ('Citations received by year, ±5', 'Existing citation fields plus historical citing data if collected', 'Cached lifetime cited_by_count grouped by publication year is NOT citations received that year.'),
    ('Self-citation rate', 'award_to_citing_edges.csv + citing_papers.csv + award authorships', 'Junior prototype exists; enforce window/denominator, extend to all authors and validate IDs/joins.'),
    ('DBLP non-award comparison pool', 'dblp_papers_raw.csv + matching checkpoint', 'Candidate pool only. Finish matching, exclude award papers, then define/control author contamination.')
], columns=['requirement', 'starting_point', 'remaining_validation_or_work'])
display(requirements)

,requirement,starting_point,remaining_validation_or_work
0,Conference/year/paper,02_award_authors_long.csv; huang_matched_openalex.csv,Join keys exist in reviewed workflow; verify cohort and unresolved paper IDs.
1,Authors,02_award_authors_long.csv; 02_author_profiles.csv,Author-paper rows are not unique authors. Resolve missing/deprecated IDs.
2,Seniority at award,02_award_authors_with_career_stage.csv,Labels exist; audit suspicious first-publication years and negative career ages.
3,Institution at award,Nested award-paper authorships; legacy junior institutions field,"Not retained in journal flattened table. Paper affiliations are a proxy, not current author affi..."
4,Authorship position,02_award_authors_long.csv; legacy position fields,First/middle/last exists; exact order needs extraction from the paper authorship list.
5,"Publication totals by year, ±5",04_all_works_cache.pkl; 04_trajectory_long.csv,"Recompute per reference event. Before/after summary notebook uses ±3, not ±5."
6,"Citations received by year, ±5",Existing citation fields plus historical citing data if collected,Cached lifetime cited_by_count grouped by publication year is NOT citations received that year.
7,Self-citation rate,award_to_citing_edges.csv + citing_papers.csv + award authorships,"Junior prototype exists; enforce window/denominator, extend to all authors and validate IDs/joins."
8,DBLP non-award comparison pool,dblp_papers_raw.csv + matching checkpoint,"Candidate pool only. Finish matching, exclude award papers, then define/control author contamina..."


## 8. Inspect a particular file or field

Change the path/field below to inspect metadata coverage. Previews use an explicit column allowlist and do not show raw JSON, pickle contents or error messages that may contain API keys. The inventory is not a security scan; review exported schemas before sharing them.

In [10]:
INSPECT_PATH = 'conf_paper/conf_data/02_award_authors_with_career_stage.csv'
FIELD_SEARCH = 'citation'  # e.g. institution, position, cited, pub, senior

display(columns_df.loc[columns_df.path.eq(INSPECT_PATH)])
display(columns_df.loc[columns_df.column.str.contains(FIELD_SEARCH, case=False, regex=False)])
SAFE_PREVIEW = {'conference', 'year', 'award_year', 'paper_title', 'title', 'author_id',
                'author_name', 'author_position', 'career_stage', 'career_age_at_award',
                'paper_openalex_id', 'openalex_id', 'dblp_row_id', 'match_status'}
inspect_df = load_csv(INSPECT_PATH)
if inspect_df is not None:
    display(inspect_df[[c for c in inspect_df.columns if c in SAFE_PREVIEW]].head(8))

,path,column,rows,nonempty_rows,missing_or_blank_rows,nonempty_pct
34,conf_paper/conf_data/02_award_authors_with_career_stage.csv,paper_openalex_id,3150,3150,0,100.00
35,conf_paper/conf_data/02_award_authors_with_career_stage.csv,paper_title,3150,3150,0,100.00
36,conf_paper/conf_data/02_award_authors_with_career_stage.csv,conference,3150,3150,0,100.00
37,conf_paper/conf_data/02_award_authors_with_career_stage.csv,award_year,3150,3150,0,100.00
38,conf_paper/conf_data/02_award_authors_with_career_stage.csv,author_id,3150,3150,0,100.00
39,conf_paper/conf_data/02_award_authors_with_career_stage.csv,author_name,3150,3150,0,100.00
40,conf_paper/conf_data/02_award_authors_with_career_stage.csv,author_position,3150,3150,0,100.00
41,conf_paper/conf_data/02_award_authors_with_career_stage.csv,first_pub_year,3150,3141,9,99.71
42,conf_paper/conf_data/02_award_authors_with_career_stage.csv,works_count,3150,3141,9,99.71
43,conf_paper/conf_data/02_award_authors_with_career_stage.csv,cited_by_count,3150,3141,9,99.71


,path,column,rows,nonempty_rows,missing_or_blank_rows,nonempty_pct
74,conf_paper/conf_data/04_author_before_after_lift.csv,citation_lift,2700,2700,0,100.00
92,conf_paper/conf_data/04_trajectory_long.csv,citations,23721,23721,0,100.00
303,data/figures/16_conference_quality.csv,citation_ratio,28,20,8,71.43
365,data/figures/summary_stats.csv,median_citations,30,30,0,100.00
381,data/matched/author_lift.csv,pre_avg_citations,784,784,0,100.00
382,data/matched/author_lift.csv,post_avg_citations,784,762,22,97.19
385,data/matched/author_lift.csv,lift_citations,784,762,22,97.19
598,data/matched/nonaward_lift.csv,pre_avg_citations,1372,1372,0,100.00
599,data/matched/nonaward_lift.csv,post_avg_citations,1372,1372,0,100.00
602,data/matched/nonaward_lift.csv,lift_citations,1372,1116,256,81.34


,paper_openalex_id,paper_title,conference,award_year,author_id,author_name,author_position,career_age_at_award,career_stage
0,https://openalex.org/W2788603415,Memory-Augmented Monte Carlo Tree Search,AAAI,2018,https://openalex.org/A5026521600,Chenjun Xiao,first,6.0,Senior
1,https://openalex.org/W2788603415,Memory-Augmented Monte Carlo Tree Search,AAAI,2018,https://openalex.org/A5014823249,Jincheng Mei,middle,4.0,Junior
2,https://openalex.org/W2788603415,Memory-Augmented Monte Carlo Tree Search,AAAI,2018,https://openalex.org/A5010677076,Martin Müller,last,52.0,Senior
3,https://openalex.org/W2798397965,Finding syntax in human encephalography with beam search,ACL,2018,https://openalex.org/A5054978902,John Hale,first,88.0,Senior
4,https://openalex.org/W2798397965,Finding syntax in human encephalography with beam search,ACL,2018,https://openalex.org/A5111222692,Chris Dyer,middle,48.0,Senior
5,https://openalex.org/W2798397965,Finding syntax in human encephalography with beam search,ACL,2018,https://openalex.org/A5069646529,Adhiguna Kuncoro,middle,2.0,Junior
6,https://openalex.org/W2798397965,Finding syntax in human encephalography with beam search,ACL,2018,https://openalex.org/A5039799789,Jonathan Brennan,last,16.0,Senior
7,https://openalex.org/W2963033005,Learning to Ask Good Questions: Ranking Clarification Questions using Neural Expected Value of P...,ACL,2018,https://openalex.org/A5001484153,Sudha Rao,first,7.0,Senior


## 9. Optional report export

Default: off. Enabling this cell's configuration flag writes only new inventory CSVs and a run manifest in `conf_paper/conf_data/data_inventory_reports/`. It does not alter source datasets. The manifest records the scan time and Git commit; the notebook itself contains no precomputed research results.

In [11]:
if EXPORT_REPORTS:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    reports = {'files': inventory, 'columns': columns_df, 'key_locations': ledger,
               'matching_progress': matching_progress, 'requirements': requirements}
    for name, frame in reports.items():
        frame.to_csv(REPORT_DIR / f'{stamp}_{name}.csv', index=False)
    manifest = {'run_at_utc': RUN_AT, 'git_commit': git_value(['rev-parse', 'HEAD']),
                'scan_roots': [p.relative_to(ROOT).as_posix() for p in SCAN_ROOTS],
                'csv_chunk_size': CHUNK_SIZE, 'hash_files': HASH_FILES,
                'source_files_modified': False, 'external_api_calls': False}
    (REPORT_DIR / f'{stamp}_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Wrote inventory reports only:', REPORT_DIR)
else:
    print('Export disabled: no repository files were written.')

Export disabled: no repository files were written.
